In [1]:
# !pip install selenium beautifulsoup4 pillow tqdm requests

In [2]:
import os
import re
import time
import hashlib
import requests

from pathlib import Path
from io import BytesIO
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from PIL import Image
from IPython.display import IFrame, display

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

import cv2
import yt_dlp
import numpy as np
from pathlib import Path

In [3]:
# --- GLOBAL DATASET CONFIGURATION ---
MASTER_DATASET_DIR = Path("ipl_2023_master_dataset")
MASTER_DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Master directory established at: {MASTER_DATASET_DIR.resolve()}")

Master directory established at: /Users/eklavyabhardwaj/Projects/IITBombayGroupPorject/ipl_2023_master_dataset


In [4]:
START_URL = "https://www.espncricinfo.com/series/indian-premier-league-2023-1345038/photo"
BASE_URL = "https://www.espncricinfo.com"

OUTPUT_DIR = MASTER_DATASET_DIR / "ipl2023"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Referer": "https://www.espncricinfo.com/",
}

In [5]:
def safe_name(text, max_len=120):
    text = str(text)
    text = re.sub(r"[^\w\s.-]", "", text)
    text = re.sub(r"\s+", "_", text.strip())
    return text[:max_len] or "album"


def get_rendered_html(url, scroll_times=5, wait=2, show_browser=True):
    options = Options()

    if not show_browser:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(f"user-agent={headers['User-Agent']}")

    driver = webdriver.Chrome(options=options)

    try:
        driver.get(url)
        time.sleep(wait)

        last_height = 0

        for _ in range(scroll_times):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(wait)

            height = driver.execute_script("return document.body.scrollHeight")

            if height == last_height:
                break

            last_height = height

        html = driver.page_source

        print("Loaded:", url)
        print("Final URL:", driver.current_url)
        print("Title:", driver.title)
        print("HTML length:", len(html))

        return html

    finally:
        driver.quit()


def clean_image_url(url):
    url = url.replace("\\u002F", "/")
    url = url.replace("\\/", "/")
    url = url.strip()
    return url


def extract_image_urls_from_html(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    image_urls = set()

    # Extract from img tags
    for img in soup.find_all("img"):
        for attr in ["src", "data-src", "srcset", "data-srcset"]:
            value = img.get(attr)

            if not value:
                continue

            if "srcset" in attr:
                parts = [p.strip().split(" ")[0] for p in value.split(",")]
            else:
                parts = [value]

            for part in parts:
                full = urljoin(page_url, part)
                full = clean_image_url(full)

                if (
                    re.search(r"\.(jpg|jpeg|png|webp)(\?|$)", full, re.I)
                    or "hscicdn.com/image/upload" in full
                    or "p.imgci.com" in full
                ):
                    image_urls.add(full)

    # Extract raw image URLs from scripts / JSON
    patterns = [
        r'https?://[^"\'\s<>]+?\.(?:jpg|jpeg|png|webp)(?:\?[^"\'\s<>]*)?',
        r'https?://img\d*\.hscicdn\.com/image/upload/[^"\'\s<>]+',
        r'https?://p\.imgci\.com/[^"\'\s<>]+',
    ]

    for pattern in patterns:
        for match in re.findall(pattern, html, re.I):
            image_urls.add(clean_image_url(match))

    # Remove logos/icons/small assets
    bad_keywords = [
        "logo",
        "favicon",
        "icon",
        "placeholder",
        "sprite",
        "apple-touch",
        "default-team-logo",
    ]

    filtered = set()

    for img_url in image_urls:
        lower = img_url.lower()

        if any(bad in lower for bad in bad_keywords):
            continue

        filtered.add(img_url)

    return filtered


def is_valid_image(content, min_width=400, min_height=250):
    try:
        img = Image.open(BytesIO(content))
        width, height = img.size
        return width >= min_width and height >= min_height
    except Exception:
        return False


def download_image(img_url, folder, index):
    try:
        r = requests.get(img_url, headers=headers, timeout=30)
        r.raise_for_status()

        content = r.content

        # --- NEW ASPECT RATIO/RESOLUTION CHECK ---
        try:
            with Image.open(BytesIO(content)) as img:
                width, height = img.size
                # Check for minimum 800x600
                if width < 800 or height < 600:
                    return False # Skip this image
        except Exception:
            # If PIL cannot open it, it might not be a valid image
            return False

        if not is_valid_image(content):
            return False

        digest = hashlib.md5(content).hexdigest()

        ext = os.path.splitext(urlparse(img_url).path)[1].lower()

        if ext not in [".jpg", ".jpeg", ".png", ".webp"]:
            ext = ".jpg"

        filename = folder / f"{index:04d}_{digest[:8]}{ext}"

        if filename.exists():
            return False

        with open(filename, "wb") as f:
            f.write(content)

        return True

    except Exception as e:
        print("\nFailed:", img_url, "->", e)
        return False


def get_page_title(html, fallback="album"):
    soup = BeautifulSoup(html, "html.parser")

    h1 = soup.find("h1")
    title_tag = soup.find("title")

    if h1:
        return h1.get_text(" ", strip=True)

    if title_tag:
        title = title_tag.get_text(" ", strip=True)
        title = title.replace("| ESPNcricinfo", "").strip()
        return title

    return fallback

In [6]:
## You can show_browse = True to check the chrome based web automation which is happening to render the cricinfo page

In [7]:
main_html = get_rendered_html(
    START_URL,
    scroll_times=8,
    wait=2,
    show_browser=True
)

with open("espn_ipl_photo_page_rendered.html", "w", encoding="utf-8") as f:
    f.write(main_html)

print("Saved main rendered HTML.")

Loaded: https://www.espncricinfo.com/series/indian-premier-league-2023-1345038/photo
Final URL: https://www.espncricinfo.com/series/indian-premier-league-2023-1345038/photo
Title: Indian Premier League 2023 Photos | Latest Match Pictures & Action Shots
HTML length: 903028
Saved main rendered HTML.


In [8]:
soup = BeautifulSoup(main_html, "html.parser")

all_links = []
for a in soup.find_all("a", href=True):
    text = a.get_text(" ", strip=True)
    link = urljoin(BASE_URL, a["href"])
    all_links.append((text, link))

print("Total links:", len(all_links))

print("\nSample links:")
for text, link in all_links[:30]:
    print(text, "->", link)

imgs = extract_image_urls_from_html(main_html, START_URL)

print("\nImages found directly in main page:", len(imgs))

for img_url in list(sorted(imgs))[:20]:
    print(img_url)

Total links: 78

Sample links:
 -> https://www.espncricinfo.com/
Live Scores -> https://www.espncricinfo.com/live-cricket-score
IPL 2026 -> https://www.espncricinfo.com/series/indian-premier-league-2025-26-1510719
Series -> https://www.espncricinfo.com/cricket-fixtures
Teams -> https://www.espncricinfo.com/team
News -> https://www.espncricinfo.com/cricket-news
Features -> https://www.espncricinfo.com/cricket-features
Videos -> https://www.espncricinfo.com/cricket-videos/
Stats -> https://www.espncricinfo.com/records
Home -> https://www.espncricinfo.com/series/indian-premier-league-2023-1345038
Fixtures and Results -> https://www.espncricinfo.com/series/indian-premier-league-2023-1345038/match-schedule-fixtures-and-results
Table -> https://www.espncricinfo.com/series/indian-premier-league-2023-1345038/points-table-standings
MVP -> https://www.espncricinfo.com/series/indian-premier-league-2023-1345038/most-valuable-players
Videos -> https://www.espncricinfo.com/series/indian-premier-leag

In [9]:
folder = OUTPUT_DIR
folder.mkdir(parents=True, exist_ok=True)


imgs = extract_image_urls_from_html(main_html, START_URL)

print("Images found:", len(imgs))

saved = 0
img_list = sorted(imgs)
total_imgs = len(img_list)

for i, img_url in enumerate(img_list, start=1):
    ok = download_image(img_url, folder, saved + 1)

    if ok:
        saved += 1

    print(
        f"\rDownloaded {saved}/{total_imgs} images | Checked {i}/{total_imgs}",
        end=""
    )

print()
print("Images saved:", saved)
print("Folder:", folder.resolve())

Images found: 257
Downloaded 239/257 images | Checked 256/257
Failed: https://www.espncricinfo.com/static/images/espncricinfo-og.png -> 403 Client Error: Forbidden for url: https://www.espncricinfo.com/static/images/espncricinfo-og.png
Downloaded 239/257 images | Checked 257/257
Images saved: 239
Folder: /Users/eklavyabhardwaj/Projects/IITBombayGroupPorject/ipl_2023_master_dataset/ipl2023


In [10]:
for folder in OUTPUT_DIR.iterdir():
    if folder.is_dir():
        image_files = (
            list(folder.glob("*.jpg"))
            + list(folder.glob("*.jpeg"))
            + list(folder.glob("*.png"))
            + list(folder.glob("*.webp"))
        )

        print(folder.name, "->", len(image_files), "images")

In [11]:
# import shutil

# shutil.make_archive("ipl_dataset_2023", "zip", "ipl_dataset_2023")
# print("Created ipl_dataset_espn.zip")

In [12]:
# --- CRICBUZZ CONFIGURATION ---
CB_START_URL = "https://www.cricbuzz.com/cricket-series/5945/indian-premier-league-2023/photos"
CB_BASE_URL = "https://www.cricbuzz.com"
CB_OUTPUT_DIR = MASTER_DATASET_DIR / "ipl2023"
CB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Render the HTML (Scrolling is important for lazy-loaded images)
cb_main_html = get_rendered_html(CB_START_URL, scroll_times=10, wait=2, show_browser=True)

# 2. Extract Image URLs using your existing function
cb_imgs = extract_image_urls_from_html(cb_main_html, CB_START_URL)
print(f"Images found on Cricbuzz: {len(cb_imgs)}")

# 3. Download the Images
cb_folder = CB_OUTPUT_DIR
cb_folder.mkdir(parents=True, exist_ok=True)

cb_saved = 0
cb_img_list = sorted(cb_imgs)
cb_total_imgs = len(cb_img_list)

for i, img_url in enumerate(cb_img_list, start=1):
    ok = download_image(img_url, cb_folder, cb_saved + 1)
    if ok:
        cb_saved += 1
    else:
        # This will reveal what is being skipped
        print(f"\nSkipped: {img_url}") 
        
    print(f"\rDownloaded {cb_saved}/{cb_total_imgs} images | Checked {i}/{cb_total_imgs}", end="")

print(f"\nCricbuzz Images saved: {cb_saved}")
print(f"Folder: {cb_folder.resolve()}")

Loaded: https://www.cricbuzz.com/cricket-series/5945/indian-premier-league-2023/photos
Final URL: https://www.cricbuzz.com/cricket-series/5945/indian-premier-league-2023/photos
Title: IPL | Indian Premier League 2023 photo galleries | Cricbuzz.com
HTML length: 355465
Images found on Cricbuzz: 155

Skipped: https://static.cricbuzz.com/a/img/v1/100x80/i1/c323927/srh-vs-rcb-match-65-ipl-2023.jpg
Downloaded 0/155 images | Checked 1/155
Skipped: https://static.cricbuzz.com/a/img/v1/100x80/i1/c325265/pbks-vs-rr-match-66-ipl-2023.jpg
Downloaded 0/155 images | Checked 2/155
Skipped: https://static.cricbuzz.com/a/img/v1/100x80/i1/c326287/kkr-vs-lsg-match-68-ipl-2023.jpg
Downloaded 0/155 images | Checked 3/155
Skipped: https://static.cricbuzz.com/a/img/v1/100x80/i1/c326399/dc-v-csk-match-67-ipl-2023.jpg
Downloaded 0/155 images | Checked 4/155
Skipped: https://static.cricbuzz.com/a/img/v1/100x80/i1/c328095/mi-vs-srh-match-69-ipl-2023.jpg
Downloaded 0/155 images | Checked 5/155
Skipped: https://st

In [13]:
# !pip install yt-dlp moviepy

In [14]:
# import yt_dlp

# def get_thumbnails_from_search(search_query, max_results=10):
#     """
#     Searches YouTube and returns a list of high-res thumbnail URLs.
#     """
#     ydl_opts = {
#         'extract_flat': True,  # Extracts metadata quickly without loading the full video stream
#         'skip_download': True,
#         'quiet': True
#     }
    
#     search_url = f"ytsearch{max_results}:{search_query}"
#     thumbnail_urls = []
    
#     with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#         result = ydl.extract_info(search_url, download=False)
#         if 'entries' in result:
#             for entry in result['entries']:
#                 if entry.get('thumbnails'):
#                     # The last item in the list is typically the highest resolution (hqdefault or maxresdefault)
#                     best_thumb = entry['thumbnails'][-1]['url']
#                     thumbnail_urls.append(best_thumb)
                    
#     return thumbnail_urls

# # --- Execution ---
# thumbnail_links = get_thumbnails_from_search("IPL 2022", max_results=10)
# print(f"Found {len(thumbnail_links)} thumbnail URLs.")
# # You can now loop through thumbnail_links and feed them straight into your download_image() function!

In [15]:
import time
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

# --- EXPLICIT IPLT20 ALBUM CONFIGURATION ---
IPLT20_BASE_URL = "https://www.iplt20.com"
ALBUM_START = 856
ALBUM_END = 1046

try:
    iplt20_folder = MASTER_DATASET_DIR / "ipl2023"
except NameError:
    iplt20_folder = Path("ipl_2022_master_dataset") / "iplt20_raw"

iplt20_folder.mkdir(parents=True, exist_ok=True)

def extract_gallery_images(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    img_urls = set()
    
    # Removed "bg" and "default" from the bad words list just in case 
    # IPL uses those words in valid match photo URLs
    bad_words = ["logo", "icon", "favicon", "banner", "svg"]
    
    for img in soup.find_all('img'):
        url = img.get('data-src') or img.get('src')
        if url:
            clean_url = url.split('?')[0] 
            full_url = urljoin(base_url, clean_url)
            
            if not any(bad in full_url.lower() for bad in bad_words):
                img_urls.add(full_url)
                
    return list(img_urls)

# --- BROWSER SETUP & ITERATION ---
print(f"Launching VISIBLE browser to fetch IPLT20 albums {ALBUM_START} through {ALBUM_END}...")
options = Options()
# 1. DISABLED HEADLESS MODE
# options.add_argument("--headless") 

# 2. Add an argument to disguise Selenium slightly
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(options=options)

total_iplt20_saved = 0

try:
    for album_id in range(ALBUM_START, ALBUM_END + 1):
        album_url = f"{IPLT20_BASE_URL}/photos/{album_id}"
        print(f"\nProcessing Album {album_id}: {album_url}")
        
        driver.get(album_url)
        
        # --- ANTI-BOT PAUSE LOGIC ---
        # Waits up to 15 seconds for at least one image with 'photo' in its class/src to load.
        # If a CAPTCHA appears, you have 15 seconds to click it manually in the Chrome window!
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "img"))
            )
        except:
            print("  -> Timeout: Page took too long or got stuck on a security check.")
        
        time.sleep(2)
        
        # --- DEEP SCROLLING LOGIC ---
        last_height = driver.execute_script("return document.body.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5) 
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break 
            last_height = new_height
            
        html = driver.page_source
        imgs = extract_gallery_images(html, IPLT20_BASE_URL)
        
        print(f"  -> Found {len(imgs)} potential images. Downloading...")
        
        # --- DIAGNOSTIC PRINT (Will help us fix the parser if it still returns 0) ---
        if len(imgs) == 0:
            print(f"  -> DEBUG: Current page title is: '{driver.title}'")
        
        album_saved = 0
        for img_url in imgs:
            ok = download_image(img_url, iplt20_folder, total_iplt20_saved + 1)
            if ok:
                total_iplt20_saved += 1
                album_saved += 1
                
        print(f"  -> Success! Saved {album_saved} valid match photos from Album {album_id}.")

except Exception as e:
    print(f"\nAn error occurred during extraction: {e}")
finally:
    driver.quit()

print(f"\n=========================================")
print(f"Extraction Complete!")
print(f"Total IPLT20 2022 Images saved: {total_iplt20_saved}")
print(f"Destination Folder: {iplt20_folder.resolve()}")
print(f"=========================================")

Launching VISIBLE browser to fetch IPLT20 albums 856 through 1046...

Processing Album 856: https://www.iplt20.com/photos/856
  -> Found 154 potential images. Downloading...
  -> Success! Saved 149 valid match photos from Album 856.

Processing Album 857: https://www.iplt20.com/photos/857
  -> Found 71 potential images. Downloading...
  -> Success! Saved 66 valid match photos from Album 857.

Processing Album 858: https://www.iplt20.com/photos/858
  -> Found 116 potential images. Downloading...
  -> Success! Saved 111 valid match photos from Album 858.

Processing Album 859: https://www.iplt20.com/photos/859
  -> Found 141 potential images. Downloading...
  -> Success! Saved 136 valid match photos from Album 859.

Processing Album 860: https://www.iplt20.com/photos/860
  -> Found 197 potential images. Downloading...
  -> Success! Saved 192 valid match photos from Album 860.

Processing Album 861: https://www.iplt20.com/photos/861
  -> Found 128 potential images. Downloading...
  -> Suc